# 03 — Exportación de datos para Power BI
## easyMoney | TFM Data Science & AI — Nuclio School

**Prerequisitos:** `01-eda.ipynb` y `02-eda-deep-dive.ipynb` ejecutados

**Objetivo:** Exportar los datos limpios (sin anomalías) en formato CSV
para construir el dashboard de BI en Power BI Desktop.

**Input:** `master_df_flags.parquet` — tabla maestra con flags de calidad

**Outputs:**
- `period_summary.csv` — evolución temporal de KPIs
- `product_penetration.csv` — penetración por producto y período
- `client_profile.csv` — perfil demográfico Mayo 2019
- `kpi_summary.csv` — KPIs calculados por período
- `product_penetration_long.csv` — penetración por producto en formato largo para gráfico de barras
- `age_sort.csv` — orden numérico para age_group en Power BI
- `salary_sort.csv` — orden numérico para salary_group en Power BI
- `contracts_by_type.csv` — nuevas contrataciones por tipo de cliente y período
- `segment_by_period.csv` — evolución de segmentos por período
- `revenue_by_period.csv` — revenue estimado por período (precios: cuenta €10, ahorro/inversión €40, financiación €60)
- `revenue_by_product.csv` — revenue estimado por producto Mayo 2019
- `revenue_by_region.csv` — revenue estimado por región Mayo 2019

---

In [8]:
# ── 03 — Exportación de datos para Power BI ───────────────────────────────
import pandas as pd
import os

BASE = 'C:\\Users\\farno\\OneDrive\\Desktop\\Data science & AI - Nuclio School\\proyecto final TFM\\tfm-fintech-easymoney\\data\\processed\\'

DATA_PATH   = BASE + 'master_df_flags.parquet'
OUTPUT_PATH = BASE + 'powerbi\\'

df = pd.read_parquet(DATA_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)

product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

last_partition = df['pk_partition'].max()

# ── Verificación ───────────────────────────────────────────────────────────
print(f"✓ Parquet cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Flags: {[c for c in df.columns if 'anomaly' in c]}")
print(f"✓ Última partición: {last_partition}")

✓ Parquet cargado: 5,962,924 filas × 37 columnas
✓ Flags: ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']
✓ Última partición: 2019-05-28 00:00:00


In [9]:
# ── Exportar CSVs para Power BI ───────────────────────────────────────────

# Tabla 1: Resumen por período
period_summary_export = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    active_clients       = ('active_customer', 'sum'),
    new_clients          = ('is_new_client', 'sum'),
    new_contracts        = ('new_contracts', 'sum'),
    avg_products         = ('total_products', 'mean'),
    clients_0_products   = ('total_products', lambda x: (x==0).sum()),
    clients_1_product    = ('total_products', lambda x: (x==1).sum()),
    clients_2plus        = ('total_products', lambda x: (x>=2).sum()),
).reset_index()
period_summary_export['pk_partition'] = period_summary_export['pk_partition'].astype(str).str[:10]
period_summary_export.to_csv(OUTPUT_PATH + 'period_summary.csv', index=False)
print(f"✓ period_summary.csv — {period_summary_export.shape}")

# Tabla 2: Penetración por producto por período
product_penetration = df.groupby('pk_partition')[product_cols].mean().mul(100).round(2).reset_index()
product_penetration['pk_partition'] = product_penetration['pk_partition'].astype(str).str[:10]
product_penetration.to_csv(OUTPUT_PATH + 'product_penetration.csv', index=False)
print(f"✓ product_penetration.csv — {product_penetration.shape}")

# Tabla 3: Perfil cliente última partición
df_clean = df[~df[['age_anomaly','deceased_anomaly','entry_date_anomaly']].any(axis=1)]
client_profile = df_clean[df_clean['pk_partition'] == last_partition][[
    'pk_cid', 'segment', 'age_group', 'salary_group',
    'gender', 'region_code', 'country_id',
    'total_products', 'is_new_client',
    'client_age_months', 'active_customer'
] + product_cols].copy()
client_profile['pk_partition'] = str(last_partition)[:10]
client_profile.to_csv(OUTPUT_PATH + 'client_profile.csv', index=False)
print(f"✓ client_profile.csv — {client_profile.shape}")

# Tabla 4: KPIs resumen
kpi_summary = period_summary_export.copy()
kpi_summary['pct_new_clients'] = (kpi_summary['new_clients'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_0_products']  = (kpi_summary['clients_0_products'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_1_product']   = (kpi_summary['clients_1_product'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_crosssell']   = (kpi_summary['clients_2plus'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary.to_csv(OUTPUT_PATH + 'kpi_summary.csv', index=False)
print(f"✓ kpi_summary.csv — {kpi_summary.shape}")

print(f"\n✓ Todos los archivos exportados en: {OUTPUT_PATH}")
print(f"\nArchivos Power BI:")
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<35} {size:.1f} KB")

✓ period_summary.csv — (17, 9)
✓ product_penetration.csv — (17, 15)
✓ client_profile.csv — (442157, 26)
✓ kpi_summary.csv — (17, 13)

✓ Todos los archivos exportados en: C:\Users\farno\OneDrive\Desktop\Data science & AI - Nuclio School\proyecto final TFM\tfm-fintech-easymoney\data\processed\powerbi\

Archivos Power BI:
  client_profile.csv                  42458.3 KB
  kpi_summary.csv                     1.8 KB
  period_summary.csv                  1.4 KB
  product_penetration.csv             1.5 KB
  product_penetration_long.csv        0.5 KB


In [10]:
# ── Tabla 5: Penetración por producto (formato largo para Power BI) ────────

label_map = {
    'em_acount': 'Cuenta easyMoney',
    'payroll': 'Domiciliaciones',
    'em_account_p': 'Cuenta easyMoney+',
    'debit_card': 'Tarjeta débito',
    'credit_card': 'Tarjeta crédito',
    'payroll_account': 'Cuenta nómina',
    'emc_account': 'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit': 'Depósito L/P',
    'pension_plan': 'Plan pensiones',
    'funds': 'Fondos inversión',
    'securities': 'Valores',
    'mortgage': 'Hipoteca',
    'loans': 'Préstamos'
}

df_last = df[df['pk_partition'] == last_partition]

penetration_long = pd.DataFrame({
    'producto': list(label_map.values()),
    'nombre_tecnico': list(label_map.keys()),
    'penetracion_pct': [df_last[col].mean() * 100 for col in label_map.keys()]
}).round(2).sort_values('penetracion_pct', ascending=False)

penetration_long.to_csv(OUTPUT_PATH + 'product_penetration_long.csv', index=False)

print(f"✓ product_penetration_long.csv — {penetration_long.shape}")
print(f"\nVista previa:")
print(penetration_long.to_string(index=False))

✓ product_penetration_long.csv — (14, 3)

Vista previa:
         producto     nombre_tecnico  penetracion_pct
 Cuenta easyMoney          em_acount            66.90
   Tarjeta débito         debit_card             9.77
    Cuenta nómina    payroll_account             5.99
    Cuenta Crypto        emc_account             5.59
   Plan pensiones       pension_plan             3.92
  Domiciliaciones            payroll             3.69
     Depósito L/P  long_term_deposit             1.38
  Tarjeta crédito        credit_card             1.08
          Valores         securities             0.40
 Fondos inversión              funds             0.30
        Préstamos              loans             0.01
         Hipoteca           mortgage             0.01
Cuenta easyMoney+       em_account_p             0.00
     Depósito C/P short_term_deposit             0.00


In [12]:
# ── Tablas adicionales para Power BI ──────────────────────────────────────

# Tabla 6: Sort orders para age_group y salary_group
age_sort = pd.DataFrame({
    'age_group': ['<18','18-24','25-35','35-45','45-55','55-65','65+'],
    'age_sort':  [1, 2, 3, 4, 5, 6, 7]
})
age_sort.to_csv(OUTPUT_PATH + 'age_sort.csv', index=False)
print(f"✓ age_sort.csv — {age_sort.shape}")

salary_sort = pd.DataFrame({
    'salary_group': ['sin_ingreso','<20k','20-40k','40-60k','60-80k','80-120k','120k+'],
    'salary_sort':  [1, 2, 3, 4, 5, 6, 7]
})
salary_sort.to_csv(OUTPUT_PATH + 'salary_sort.csv', index=False)
print(f"✓ salary_sort.csv — {salary_sort.shape}")

# Tabla 7: Nuevos vs existentes por período
contracts_type = df[df['pk_partition'] != df['pk_partition'].min()].groupby(
    ['pk_partition', 'is_new_client']
).agg(
    contracts=('new_contracts', 'sum'),
    client_count=('pk_cid', 'count')
).reset_index()
contracts_type['pk_partition'] = contracts_type['pk_partition'].astype(str).str[:10]
contracts_type['tipo'] = contracts_type['is_new_client'].map({0: 'Existente', 1: 'Nuevo'})
contracts_type.to_csv(OUTPUT_PATH + 'contracts_by_type.csv', index=False)
print(f"✓ contracts_by_type.csv — {contracts_type.shape}")

# Tabla 8: Segmentos por período
segment_period = df.groupby(['pk_partition', 'segment'])['pk_cid'].count().reset_index()
segment_period.columns = ['pk_partition', 'segment', 'clientes']
segment_period['pk_partition'] = segment_period['pk_partition'].astype(str).str[:10]
segment_period.to_csv(OUTPUT_PATH + 'segment_by_period.csv', index=False)
print(f"✓ segment_by_period.csv — {segment_period.shape}")

print(f"\n✓ Todos los archivos Power BI:")
for f in sorted(os.listdir(OUTPUT_PATH)):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<40} {size:.1f} KB")

✓ age_sort.csv — (7, 2)
✓ salary_sort.csv — (7, 2)
✓ contracts_by_type.csv — (32, 5)
✓ segment_by_period.csv — (51, 3)

✓ Todos los archivos Power BI:
  age_sort.csv                             0.1 KB
  client_profile.csv                       42458.3 KB
  contracts_by_type.csv                    1.1 KB
  kpi_summary.csv                          1.8 KB
  period_summary.csv                       1.4 KB
  product_penetration.csv                  1.5 KB
  product_penetration_long.csv             0.5 KB
  salary_sort.csv                          0.1 KB
  segment_by_period.csv                    1.7 KB


## Criterio de precios para revenue estimado

En ausencia de datos de margen neto, se aplican los precios fijos
definidos por Carol para la campaña de email:

| Familia | Productos | Precio unitario |
|---|---|---|
| Cuenta | em_acount, emc_account, payroll_account, payroll, debit_card, em_account_p | €10 |
| Ahorro / Inversión | short_term_deposit, long_term_deposit, funds, securities, pension_plan, credit_card | €40 |
| Financiación | loans, mortgage | €60 |

> **Revenue total estimado Mayo 2019: €5,331,300**
> **Revenue por cliente: €12.03**

In [13]:
# ── Tabla 9 & 10: Revenue estimado ────────────────────────────────────────

price_map = {
    'em_acount': 10,
    'emc_account': 10,
    'payroll_account': 10,
    'payroll': 10,
    'debit_card': 10,
    'em_account_p': 10,
    'short_term_deposit': 40,
    'long_term_deposit': 40,
    'funds': 40,
    'securities': 40,
    'pension_plan': 40,
    'credit_card': 40,
    'loans': 60,
    'mortgage': 60
}

# Revenue por período
revenue_period = []
for partition, group in df.groupby('pk_partition'):
    total_rev = sum(group[col].sum() * price for col, price in price_map.items())
    revenue_period.append({
        'pk_partition': str(partition)[:10],
        'revenue_estimado': round(total_rev, 2),
        'total_clientes': len(group),
        'revenue_por_cliente': round(total_rev / len(group), 2)
    })

df_revenue = pd.DataFrame(revenue_period)
df_revenue.to_csv(OUTPUT_PATH + 'revenue_by_period.csv', index=False)
print(f"✓ revenue_by_period.csv — {df_revenue.shape}")
print(df_revenue.to_string(index=False))

# Revenue por producto última partición
df_last = df[df['pk_partition'] == last_partition]
revenue_product = pd.DataFrame({
    'producto': list(label_map.values()),
    'nombre_tecnico': list(label_map.keys()),
    'precio_unitario': [price_map.get(col, 10) for col in label_map.keys()],
    'clientes_con_producto': [int(df_last[col].sum()) for col in label_map.keys()],
}).assign(
    revenue_estimado=lambda x: x['clientes_con_producto'] * x['precio_unitario']
).sort_values('revenue_estimado', ascending=False)

revenue_product.to_csv(OUTPUT_PATH + 'revenue_by_product.csv', index=False)
print(f"\n✓ revenue_by_product.csv — {revenue_product.shape}")
print(revenue_product.to_string(index=False))

# Revenue por región última partición
revenue_region = df_last.copy()
revenue_region['revenue'] = sum(
    revenue_region[col] * price for col, price in price_map.items()
)
revenue_region = revenue_region.groupby('region_code').agg(
    clientes=('pk_cid', 'count'),
    revenue_estimado=('revenue', 'sum')
).reset_index().sort_values('revenue_estimado', ascending=False)
revenue_region['revenue_estimado'] = revenue_region['revenue_estimado'].round(2)
revenue_region.to_csv(OUTPUT_PATH + 'revenue_by_region.csv', index=False)
print(f"\n✓ revenue_by_region.csv — {revenue_region.shape}")

✓ revenue_by_period.csv — (17, 4)
pk_partition  revenue_estimado  total_clientes  revenue_por_cliente
  2018-01-28           3557180          239493                14.85
  2018-02-28           3653700          242521                15.07
  2018-03-28           3749470          245258                15.29
  2018-04-28           3833010          247463                15.49
  2018-05-28           3853860          249926                15.42
  2018-06-28           3964580          252104                15.73
  2018-07-28           4160570          339339                12.26
  2018-08-28           4270830          352922                12.10
  2018-09-28           4501120          375323                11.99
  2018-10-28           4754720          402300                11.82
  2018-11-28           4883850          416387                11.73
  2018-12-28           5020240          422481                11.88
  2019-01-28           4919290          426875                11.52
  2019-02-28  